# M1 Fault Group Lead-Time Audit Lock

28번 pre-event gate 결과를 재사용해 고장군별 stable lead-time을 잠그는 노트북이다.

## Context & Methods

새 리드타임 모델은 만들지 않는다. 28번의 로지스틱 pre-event 확률과 rolling anchor 예측만 집계한다.

In [1]:
from scripts.run_29_fault_group_leadtime_audit_lock import run_analysis
results = run_analysis(write_notebook_file=False)
results['summary'].head()

29 M1 fault group lead-time audit lock complete
       fault_group  event_count  median_stable_lead_time_days   leadtime_label
control_controller           12                           0.0 report_time_only
      pump_failure            5                           0.5     short_stable
    valve_actuator            4                           3.5     early_stable
pressure_regulator            4                           0.0 report_time_only
leakage_water_loss            4                           7.0     early_stable
    unknown_review            0                           NaN      review_only
                                                 check  pass                                                                                                         detail
    input_exists_m1_fault_rolling_leadtime_summary.csv  True                                   C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_fault_rolling_leadtime_summary.csv
input_exists_m1_fault_rolling_leadtime_predictions.csv 

,scope,fault_group,event_count,profile_event_count,included_event_ids,scope_excluded_event_ids,efd_possible_true_count,d0_detection_rate,first_crossing_event_count,first_crossing_detection_rate,...,median_priority_score,group_weight,leadtime_label,leadtime_confidence,operational_meaning,d_minus_7_detection_rate,d_minus_5_detection_rate,d_minus_3_detection_rate,d_minus_1_detection_rate,d_minus_12h_detection_rate
0,main,control_controller,12,12,3|6|23|32|36|37|44|49|52|60|62|63,,9,0.833333,11,0.916667,...,92.290698,1.000000,report_time_only,high,same_day_fault_signal,0.416667,0.25,0.416667,0.50,0.083333
1,main,pump_failure,5,5,5|13|15|24|47,,5,0.800000,5,1.000000,...,82.597526,0.636043,short_stable,high,short_leadtime_warning_candidate,0.600000,0.60,0.600000,0.80,0.400000
2,main,valve_actuator,5,5,1|11|29|53|67,,4,0.600000,5,1.000000,...,75.700359,0.580894,report_time_only,medium,same_day_fault_signal,1.000000,0.80,1.000000,1.00,0.200000
3,main,pressure_regulator,4,4,7|38|57|64,,4,1.000000,4,1.000000,...,90.924545,0.521702,report_time_only,high,same_day_fault_signal,0.750000,0.75,0.750000,0.75,0.000000
4,main,leakage_water_loss,5,5,10|20|40|45|65,,5,0.600000,5,1.000000,...,68.427544,0.444043,early_stable,medium,days_before_warning_candidate,0.800000,0.60,1.000000,0.60,0.400000


## Data

이벤트 단위 audit과 main/clean 고장군 요약을 확인한다.

In [2]:
display(results['event_audit'].head(10))
display(results['summary'])

,event_id,substation_id,fault_group,fault_label,problem_en,efd_possible,monitoring_potential,report_date,d0_probability,d0_prediction,...,anchor_d_minus_3_coverage_rate,anchor_d_minus_1_probability,anchor_d_minus_1_prediction,anchor_d_minus_1_coverage_rate,anchor_d_minus_12h_probability,anchor_d_minus_12h_prediction,anchor_d_minus_12h_coverage_rate,anchor_d0_probability,anchor_d0_prediction,anchor_d0_coverage_rate
0,3,12,control_controller,Control unit: Incorrect parameterisation,no heat,True,4.0,2015-12-01 10:56:00,0.985404,1,...,1.0,0.431514,0,1.0,0.177834,0,1.000000,0.985404,1,1.000000
1,6,21,control_controller,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2016-12-06 13:12:00,0.749192,1,...,1.0,0.731242,1,1.0,0.213363,0,1.000000,0.749192,1,1.000000
2,23,27,control_controller,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2020-01-20 12:57:00,0.898881,1,...,1.0,0.789048,1,1.0,0.025385,0,1.000000,0.898881,1,1.000000
3,32,21,control_controller,Control unit: Incorrect parameterisation,not enough heat,True,4.0,2019-10-19 10:38:00,0.860889,1,...,1.0,0.760067,1,1.0,0.069914,0,1.000000,0.860889,1,1.000000
4,36,7,control_controller,Control unit: Incorrect parameterisation,not enough heat,False,4.0,2015-04-02 20:05:00,0.424850,0,...,1.0,0.275889,0,1.0,0.574280,0,1.000000,0.424850,0,1.000000
5,37,19,control_controller,Control unit: Incorrect parameterisation,no heat,True,4.0,2018-11-21 09:15:00,0.757123,1,...,1.0,0.190848,0,1.0,0.159082,0,1.000000,0.757123,1,1.000000
6,44,8,control_controller,Temperature monitor/controller defective,no DHW,True,3.0,2018-04-27 08:15:00,0.858773,1,...,1.0,0.468795,0,1.0,0.829177,1,1.000000,0.858773,1,1.000000
7,49,18,control_controller,Control unit: Incorrect parameterisation,no DHW,True,4.0,2019-05-04 07:19:00,0.960765,1,...,1.0,0.218729,0,1.0,0.353569,0,0.997024,0.960765,1,0.997024
8,52,21,control_controller,Control unit: Incorrect parameterisation,no heat,True,4.0,2016-12-12 15:55:00,0.791321,1,...,1.0,0.466192,0,1.0,0.357170,0,1.000000,0.791321,1,1.000000
9,60,4,control_controller,Control unit: Incorrect parameterisation,no heat,False,4.0,2015-03-18 18:54:00,0.911346,1,...,1.0,0.877128,1,1.0,0.146774,0,1.000000,0.911346,1,1.000000


,scope,fault_group,event_count,profile_event_count,included_event_ids,scope_excluded_event_ids,efd_possible_true_count,d0_detection_rate,first_crossing_event_count,first_crossing_detection_rate,...,median_priority_score,group_weight,leadtime_label,leadtime_confidence,operational_meaning,d_minus_7_detection_rate,d_minus_5_detection_rate,d_minus_3_detection_rate,d_minus_1_detection_rate,d_minus_12h_detection_rate
0,main,control_controller,12,12,3|6|23|32|36|37|44|49|52|60|62|63,,9,0.833333,11,0.916667,...,92.290698,1.000000,report_time_only,high,same_day_fault_signal,0.416667,0.25,0.416667,0.50,0.083333
1,main,pump_failure,5,5,5|13|15|24|47,,5,0.800000,5,1.000000,...,82.597526,0.636043,short_stable,high,short_leadtime_warning_candidate,0.600000,0.60,0.600000,0.80,0.400000
2,main,valve_actuator,5,5,1|11|29|53|67,,4,0.600000,5,1.000000,...,75.700359,0.580894,report_time_only,medium,same_day_fault_signal,1.000000,0.80,1.000000,1.00,0.200000
3,main,pressure_regulator,4,4,7|38|57|64,,4,1.000000,4,1.000000,...,90.924545,0.521702,report_time_only,high,same_day_fault_signal,0.750000,0.75,0.750000,0.75,0.000000
4,main,leakage_water_loss,5,5,10|20|40|45|65,,5,0.600000,5,1.000000,...,68.427544,0.444043,early_stable,medium,days_before_warning_candidate,0.800000,0.60,1.000000,0.60,0.400000
5,main,unknown_review,2,2,34|69,,2,0.500000,2,1.000000,...,54.437974,0.100000,review_only,review,manual_review_only,0.500000,0.50,0.500000,1.00,0.500000
6,clean,control_controller,12,12,3|6|23|32|36|37|44|49|52|60|62|63,20|34|67|69,9,0.833333,11,0.916667,...,92.290698,1.000000,report_time_only,high,same_day_fault_signal,0.416667,0.25,0.416667,0.50,0.083333
7,clean,pump_failure,5,5,5|13|15|24|47,20|34|67|69,5,0.800000,5,1.000000,...,82.597526,0.636043,short_stable,high,short_leadtime_warning_candidate,0.600000,0.60,0.600000,0.80,0.400000
8,clean,valve_actuator,4,5,1|11|29|53,20|34|67|69,3,0.500000,4,1.000000,...,50.536374,0.580894,early_stable,medium,days_before_warning_candidate,1.000000,0.75,1.000000,1.00,0.250000
9,clean,pressure_regulator,4,4,7|38|57|64,20|34|67|69,4,1.000000,4,1.000000,...,90.924545,0.521702,report_time_only,high,same_day_fault_signal,0.750000,0.75,0.750000,0.75,0.000000


## Results

공식 리드타임은 stable crossing 기준이다.

In [3]:
display(results['decision_matrix'])
display(results['quality_audit'])

,scope,fault_group,event_count,stable_crossing_detection_rate,median_stable_lead_time_hours,median_stable_lead_time_days,leadtime_label,leadtime_confidence,operational_meaning,d0_detection_rate,group_weight,median_priority_score,included_event_ids,scope_excluded_event_ids,leadtime_lock_decision,priority_v1_usage
0,main,control_controller,12,0.833333,0.0,0.0,report_time_only,high,same_day_fault_signal,0.833333,1.000000,92.290698,3|6|23|32|36|37|44|49|52|60|62|63,,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
1,main,pump_failure,5,0.800000,12.0,0.5,short_stable,high,short_leadtime_warning_candidate,0.800000,0.636043,82.597526,5|13|15|24|47,,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
2,main,valve_actuator,5,0.600000,0.0,0.0,report_time_only,medium,same_day_fault_signal,0.600000,0.580894,75.700359,1|11|29|53|67,,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
3,main,pressure_regulator,4,1.000000,0.0,0.0,report_time_only,high,same_day_fault_signal,1.000000,0.521702,90.924545,7|38|57|64,,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
4,main,leakage_water_loss,5,0.600000,168.0,7.0,early_stable,medium,days_before_warning_candidate,0.600000,0.444043,68.427544,10|20|40|45|65,,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
5,main,unknown_review,2,0.500000,0.0,0.0,review_only,review,manual_review_only,0.500000,0.100000,54.437974,34|69,,review_only,context_only_priority_formula_unchanged
6,clean,control_controller,12,0.833333,0.0,0.0,report_time_only,high,same_day_fault_signal,0.833333,1.000000,92.290698,3|6|23|32|36|37|44|49|52|60|62|63,20|34|67|69,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
7,clean,pump_failure,5,0.800000,12.0,0.5,short_stable,high,short_leadtime_warning_candidate,0.800000,0.636043,82.597526,5|13|15|24|47,20|34|67|69,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
8,clean,valve_actuator,4,0.500000,84.0,3.5,early_stable,medium,days_before_warning_candidate,0.500000,0.580894,50.536374,1|11|29|53,20|34|67|69,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged
9,clean,pressure_regulator,4,1.000000,0.0,0.0,report_time_only,high,same_day_fault_signal,1.000000,0.521702,90.924545,7|38|57|64,20|34|67|69,group_leadtime_locked_as_audit,context_only_priority_formula_unchanged


,check,pass,detail
0,input_exists_m1_fault_rolling_leadtime_summary...,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
1,input_exists_m1_fault_rolling_leadtime_predict...,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
2,input_exists_m1_fault_dispatch_priority_v1.csv,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
3,input_exists_m1_fault_group_priority_profile.csv,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
4,input_exists_m1_fault_pre_event_v1_lock_decisi...,True,C:\3rd_Project\HeatGridAgent\07_데이터산출물\m1_faul...
5,pre_event_lock_unchanged,True,decision=fault_pre_event_gate_v1_locked_for_M1...
6,no_new_model_training,True,29 uses 28 prediction CSV outputs only
7,event_audit_33_faults,True,rows=33
8,one_fault_group_per_event,True,unique_events=33; missing_groups=0
9,special_event_flags_retained,True,present=20|34|67|69


## Takeaways

- 28번 pre-event gate는 변경하지 않는다.
- 고장군별 리드타임은 `stable_crossing_lead_time_hours` 기준으로 해석한다.
- 28번 priority score는 그대로 두고, 이번 표는 우선순위 해석 근거로 사용한다.